# 第16章 高级系统软件 - 编译器与翻译软件
## Chapter 16: Advanced System Software - Compilers & Translators

**Cambridge A-Level Computer Science (9618)**

---

### 本节学习目标 Learning Objectives

1. 理解解释器和编译器的区别与工作原理
2. 掌握编译的各个阶段：词法分析、语法分析、代码生成、代码优化
3. 理解并能使用语法图(Syntax Diagrams)
4. 理解并能使用BNF范式(Backus-Naur Form)
5. 掌握逆波兰表达式(RPN)的转换与求值
6. 能够用Python实现简单的词法分析器和RPN计算器

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.patches as mpatches
import numpy as np
import re

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False



---
## 1. 翻译软件概述 (Translation Software Overview)

### 1.1 为什么需要翻译软件？

计算机只能直接执行**机器码(Machine Code)** -- 由0和1组成的指令。
但人类用0和1编程几乎不可能，所以我们使用**高级语言(High-level Language)** 编程。

**翻译软件**的作用就是将人类可读的代码翻译成机器可执行的代码。

### 1.2 三种翻译软件

| 类型 | 英文 | 工作方式 | 举例 |
|------|------|----------|------|
| 汇编器 | Assembler | 将汇编语言翻译为机器码（一对一） | NASM, MASM |
| 编译器 | Compiler | 将整个源代码翻译为目标代码 | GCC, javac |
| 解释器 | Interpreter | 逐行翻译并执行 | Python, JavaScript |

### 1.3 编译器 vs 解释器 -- 深度对比

| 特征 | 编译器 (Compiler) | 解释器 (Interpreter) |
|------|-------------------|---------------------|
| 翻译方式 | 一次翻译整个程序 | 逐行翻译并执行 |
| 执行速度 | 快（已编译为机器码） | 慢（每次运行都要翻译） |
| 错误检测 | 编译时一次性报告所有错误 | 遇到错误就停止 |
| 输出 | 生成可执行文件 | 不生成可执行文件 |
| 调试 | 需要重新编译 | 容易调试（可以交互执行） |
| 分发 | 可以只分发可执行文件 | 必须分发源代码 + 解释器 |

**为什么Python用解释器而C用编译器？**

- Python追求**开发效率**：快速编写、测试、修改，适合原型开发和脚本
- C追求**运行效率**：操作系统、嵌入式系统需要极高的执行速度
- 实际上很多语言混合使用两者：Java先编译为字节码，再由JVM解释执行

In [ ]:
# 可视化：编译器 vs 解释器 工作流程
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# === 编译器流程 ===
compiler_steps = ['源代码\n(Source\nCode)', '编译器\n(Compiler)', '目标代码\n(Object\nCode)', '链接器\n(Linker)', '可执行文件\n(Executable)', '执行\n(Execute)']
compiler_colors = ['#3498db', '#e74c3c', '#f39c12', '#9b59b6', '#2ecc71', '#1abc9c']

for i, (step, color) in enumerate(zip(compiler_steps, compiler_colors)):
    x = 0.08 + i * 0.155
    rect = patches.FancyBboxPatch((x, 0.15), 0.12, 0.7,
                                   boxstyle='round,pad=0.03',
                                   facecolor=color, alpha=0.85, edgecolor='white', lw=2)
    ax1.add_patch(rect)
    ax1.text(x + 0.06, 0.5, step, ha='center', va='center',
             fontsize=9, fontweight='bold', color='white')
    if i < len(compiler_steps) - 1:
        ax1.annotate('', xy=(x + 0.14, 0.5), xytext=(x + 0.12, 0.5),
                     arrowprops=dict(arrowstyle='->', color='#333', lw=2))

ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.axis('off')
ax1.set_title('编译器工作流程 (Compiler Workflow) -- 一次翻译，多次执行', fontsize=12, fontweight='bold')

# === 解释器流程 ===
interp_steps = ['源代码\n第1行', '翻译+执行\n第1行', '源代码\n第2行', '翻译+执行\n第2行', '源代码\n第3行', '翻译+执行\n第3行']
interp_colors = ['#3498db', '#e74c3c', '#3498db', '#e74c3c', '#3498db', '#e74c3c']

for i, (step, color) in enumerate(zip(interp_steps, interp_colors)):
    x = 0.08 + i * 0.155
    rect = patches.FancyBboxPatch((x, 0.15), 0.12, 0.7,
                                   boxstyle='round,pad=0.03',
                                   facecolor=color, alpha=0.85, edgecolor='white', lw=2)
    ax2.add_patch(rect)
    ax2.text(x + 0.06, 0.5, step, ha='center', va='center',
             fontsize=9, fontweight='bold', color='white')
    if i < len(interp_steps) - 1:
        ax2.annotate('', xy=(x + 0.14, 0.5), xytext=(x + 0.12, 0.5),
                     arrowprops=dict(arrowstyle='->', color='#333', lw=2))

ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.axis('off')
ax2.set_title('解释器工作流程 (Interpreter Workflow) -- 逐行翻译，逐行执行', fontsize=12, fontweight='bold')

plt.suptitle('编译器 vs 解释器', fontsize=14, fontweight='bold')
plt.tight_layout()


---
## 2. 编译阶段详解 (Compilation Stages)

编译器将源代码翻译为机器码的过程分为多个阶段：

```
源代码 -> 词法分析 -> 语法分析 -> 语义分析 -> 代码优化 -> 代码生成 -> 目标代码
```

### 2.1 词法分析 (Lexical Analysis)

**做什么：** 将源代码字符串分解为**记号(Token)** -- 有意义的最小单元。

**为什么需要词法分析？**

源代码在计算机看来只是一串字符。编译器首先需要识别出哪些字符组合在一起有意义：

```
源代码:  total = price * 2 + tax
记号:    [IDENTIFIER:total] [ASSIGN:=] [IDENTIFIER:price] [OPERATOR:*] [NUMBER:2] [OPERATOR:+] [IDENTIFIER:tax]
```

词法分析还会：
- 去除空格和注释
- 识别关键字（if, while, for等）
- 构建**符号表(Symbol Table)**

**记号的类型 (Token Types)：**

| 类型 | 英文 | 举例 |
|------|------|------|
| 关键字 | Keyword | if, while, return |
| 标识符 | Identifier | x, total, myFunc |
| 常量 | Literal | 42, 3.14, "hello" |
| 运算符 | Operator | +, -, *, ==, != |
| 分隔符 | Delimiter | (, ), {, }, ; |

In [ ]:
# Python实现：简单的词法分析器 (Lexer / Tokenizer)

class Token:
    def __init__(self, type_, value):
        self.type = type_
        self.value = value

    def __repr__(self):
        return f'Token({self.type}, {self.value!r})'

class SimpleLexer:
    """简单的词法分析器"""
    KEYWORDS = {'if', 'else', 'while', 'for', 'return', 'def', 'print'}

    def __init__(self, text):
        self.text = text
        self.pos = 0
        self.tokens = []

    def tokenize(self):
        """将源代码分解为记号列表"""
        while self.pos < len(self.text):
            char = self.text[self.pos]

            # 跳过空白字符
            if char.isspace():
                self.pos += 1
                continue

            # 数字
            if char.isdigit():
                num = ''
                while self.pos < len(self.text) and (self.text[self.pos].isdigit() or self.text[self.pos] == '.'):
                    num += self.text[self.pos]
                    self.pos += 1
                self.tokens.append(Token('NUMBER', num))
                continue

            # 标识符或关键字
            if char.isalpha() or char == '_':
                word = ''
                while self.pos < len(self.text) and (self.text[self.pos].isalnum() or self.text[self.pos] == '_'):
                    word += self.text[self.pos]
                    self.pos += 1
                if word in self.KEYWORDS:
                    self.tokens.append(Token('KEYWORD', word))
                else:
                    self.tokens.append(Token('IDENTIFIER', word))
                continue

            # 运算符
            if char in '+-*/=<>!%':
                # 检查双字符运算符
                if self.pos + 1 < len(self.text) and self.text[self.pos:self.pos+2] in ('==', '!=', '<=', '>='):
                    self.tokens.append(Token('OPERATOR', self.text[self.pos:self.pos+2]))
                    self.pos += 2
                else:
                    self.tokens.append(Token('OPERATOR', char))
                    self.pos += 1
                continue

            # 分隔符
            if char in '(){}[];,:':
                self.tokens.append(Token('DELIMITER', char))
                self.pos += 1
                continue

            # 字符串
            if char in '"\'':
                quote = char
                self.pos += 1
                string = ''
                while self.pos < len(self.text) and self.text[self.pos] != quote:
                    string += self.text[self.pos]
                    self.pos += 1
                self.pos += 1  # 跳过结束引号
                self.tokens.append(Token('STRING', string))
                continue

            # 未知字符
            self.tokens.append(Token('UNKNOWN', char))
            self.pos += 1

        return self.tokens

# 测试词法分析器
test_codes = [
    'total = price * 2 + tax',
    'if x >= 10:',
    'for i in range(100):',
    'result = (a + b) * (c - d)',
    'print("hello world")',
]

for code in test_codes:
    print(f'源代码: {code}')
    lexer = SimpleLexer(code)
    tokens = lexer.tokenize()
    for token in tokens:
        print(f'  {token}')


In [ ]:
# 可视化：词法分析过程
fig, ax = plt.subplots(figsize=(14, 5))

source = 'total = price * 2 + tax'
lexer = SimpleLexer(source)
tokens = lexer.tokenize()

# 源代码显示
ax.text(0.5, 0.92, f'源代码 (Source Code):  {source}',
        ha='center', fontsize=13, fontweight='bold',
        fontfamily='monospace',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#2c3e50', alpha=0.9),
        color='#2ecc71')

ax.text(0.5, 0.72, '词法分析 (Lexical Analysis)',
        ha='center', fontsize=11, fontweight='bold', color='#666')
ax.annotate('', xy=(0.5, 0.62), xytext=(0.5, 0.68),
            arrowprops=dict(arrowstyle='->', color='#666', lw=2))

# Token颜色映射
token_colors = {
    'IDENTIFIER': '#3498db',
    'OPERATOR': '#e74c3c',
    'NUMBER': '#f39c12',
    'KEYWORD': '#9b59b6',
    'STRING': '#2ecc71',
    'DELIMITER': '#1abc9c',
}

# 显示记号
n = len(tokens)
start_x = 0.5 - (n * 0.12) / 2
for i, token in enumerate(tokens):
    x = start_x + i * 0.12
    color = token_colors.get(token.type, '#95a5a6')
    rect = patches.FancyBboxPatch((x, 0.25), 0.10, 0.35,
                                   boxstyle='round,pad=0.03',
                                   facecolor=color, alpha=0.85, edgecolor='white', lw=2)
    ax.add_patch(rect)
    ax.text(x + 0.05, 0.48, token.type, ha='center', va='center',
            fontsize=7, fontweight='bold', color='white')
    ax.text(x + 0.05, 0.36, repr(token.value), ha='center', va='center',
            fontsize=8, color='white', fontfamily='monospace')

# 图例
legend_patches = [patches.Patch(color=c, label=t) for t, c in token_colors.items()]
ax.legend(handles=legend_patches, loc='lower center', ncol=6, fontsize=8)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('词法分析过程演示 (Lexical Analysis Process)', fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()


### 2.2 语法分析 (Syntax Analysis / Parsing)

**做什么：** 检查记号序列是否符合语言的**语法规则(Grammar Rules)**，并构建**语法树(Syntax Tree / Abstract Syntax Tree, AST)**。

**为什么需要语法分析？**

词法分析只识别了单个记号，但没有检查它们的**组合是否合法**：
- `x = 5 + 3` -- 合法
- `x = + 5 3` -- 不合法（运算符位置错误）
- `= x 5 + 3` -- 不合法（赋值号位置错误）

语法分析就像检查英语句子的语法：主语-谓语-宾语的顺序必须正确。

### 语法树 (Abstract Syntax Tree, AST)

语法树是表达式结构的树形表示。例如 `(3 + 4) * 2`：

```
    *
   / \
  +   2
 / \
3   4
```

树的结构自然地表示了运算的**优先级和结合性**。

In [ ]:
# 可视化：语法树
def draw_ast(ax, expression_str, nodes, edges):
    """
    绘制抽象语法树
    nodes: dict of {label: (x, y)}
    edges: list of (parent_label, child_label)
    """
    # 画边
    for parent, child in edges:
        px, py = nodes[parent]
        cx, cy = nodes[child]
        ax.plot([px, cx], [py, cy], 'k-', lw=2, zorder=1)

    # 画节点
    for label, (x, y) in nodes.items():
        is_op = label.rstrip('_') in '+-*/='
        color = '#e74c3c' if is_op else '#3498db'
        display = label.rstrip('_')  # 去掉可能的后缀
        circle = plt.Circle((x, y), 0.06, color=color, zorder=2)
        ax.add_patch(circle)
        ax.text(x, y, display, ha='center', va='center',
                fontsize=14, fontweight='bold', color='white', zorder=3)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# 表达式1: (3 + 4) * 2
nodes1 = {
    '*': (0.5, 0.85),
    '+': (0.3, 0.55),
    '2': (0.7, 0.55),
    '3': (0.15, 0.25),
    '4': (0.45, 0.25),
}
edges1 = [('*', '+'), ('*', '2'), ('+', '3'), ('+', '4')]
draw_ast(ax1, '(3 + 4) * 2', nodes1, edges1)
ax1.set_xlim(0, 1)
ax1.set_ylim(0.05, 1.0)
ax1.set_aspect('equal')
ax1.axis('off')
ax1.set_title('语法树: (3 + 4) * 2 = 14', fontsize=12, fontweight='bold')

# 表达式2: a = b + c * d
nodes2 = {
    '=': (0.5, 0.85),
    'a': (0.25, 0.55),
    '+_': (0.7, 0.55),
    'b': (0.55, 0.25),
    '*': (0.85, 0.25),
    'c': (0.75, 0.0),
    'd': (0.95, 0.0),
}
edges2 = [('=', 'a'), ('=', '+_'), ('+_', 'b'), ('+_', '*'), ('*', 'c'), ('*', 'd')]
draw_ast(ax2, 'a = b + c * d', nodes2, edges2)
ax2.set_xlim(0, 1.1)
ax2.set_ylim(-0.15, 1.0)
ax2.set_aspect('equal')
ax2.axis('off')
ax2.set_title('语法树: a = b + c * d', fontsize=12, fontweight='bold')

plt.suptitle('抽象语法树 (Abstract Syntax Tree)', fontsize=14, fontweight='bold')
plt.tight_layout()


### 2.3 代码优化 (Code Optimisation)

**做什么：** 在不改变程序功能的前提下，改进代码使其运行更快或占用更少内存。

**常见优化技术：**

| 技术 | 英文 | 示例 |
|------|------|------|
| 常量折叠 | Constant Folding | `x = 3 + 4` -> `x = 7` |
| 死代码消除 | Dead Code Elimination | 删除永远不会执行的代码 |
| 循环不变量外提 | Loop-Invariant Code Motion | 将循环中不变的计算移到循环外 |
| 公共子表达式消除 | Common Subexpression Elimination | 避免重复计算相同表达式 |

### 2.4 代码生成 (Code Generation)

**做什么：** 将优化后的中间代码翻译为目标机器的机器码或汇编代码。

**为什么要分阶段？为什么不直接从源代码生成机器码？**

分阶段设计的好处：
1. **模块化** - 每个阶段专注于一件事，便于维护
2. **可移植** - 前端（词法/语法分析）与后端（代码生成）分离，换一个目标平台只需改后端
3. **优化空间** - 可以在中间表示上做各种优化

In [ ]:
# 可视化：编译阶段完整流程图
fig, ax = plt.subplots(figsize=(14, 7))

stages = [
    ('源代码\n(Source Code)', '#2c3e50', 0.05),
    ('词法分析\n(Lexical\nAnalysis)', '#e74c3c', 0.20),
    ('语法分析\n(Syntax\nAnalysis)', '#f39c12', 0.38),
    ('语义分析\n(Semantic\nAnalysis)', '#9b59b6', 0.56),
    ('代码优化\n(Code\nOptimisation)', '#3498db', 0.72),
    ('代码生成\n(Code\nGeneration)', '#2ecc71', 0.88),
]

outputs = [
    '字符流', '记号流\n(Tokens)', '语法树\n(AST)', '标注的AST', '优化的\n中间代码', '目标代码\n(Object Code)'
]

for i, (name, color, x) in enumerate(stages):
    # 阶段方框
    rect = patches.FancyBboxPatch((x, 0.4), 0.13, 0.4,
                                   boxstyle='round,pad=0.03',
                                   facecolor=color, alpha=0.85, edgecolor='white', lw=2)
    ax.add_patch(rect)
    ax.text(x + 0.065, 0.6, name, ha='center', va='center',
            fontsize=9, fontweight='bold', color='white')

    # 输出标签
    if i < len(outputs):
        ax.text(x + 0.065, 0.28, outputs[i], ha='center', va='center',
                fontsize=8, color='#666', style='italic')

    # 箭头
    if i < len(stages) - 1:
        next_x = stages[i+1][2]
        ax.annotate('', xy=(next_x, 0.6), xytext=(x + 0.13, 0.6),
                     arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# 符号表（贯穿所有阶段）
rect = patches.FancyBboxPatch((0.20, 0.05), 0.65, 0.12,
                               boxstyle='round,pad=0.03',
                               facecolor='#ecf0f1', edgecolor='#bdc3c7', lw=2)
ax.add_patch(rect)
ax.text(0.525, 0.11, '符号表 (Symbol Table) -- 贯穿所有阶段', ha='center', va='center',
        fontsize=10, fontweight='bold', color='#7f8c8d')

# 虚线连接符号表和各阶段
for _, _, x in stages[1:]:
    ax.plot([x + 0.065, x + 0.065], [0.17, 0.4], '--', color='#bdc3c7', lw=1)

ax.set_xlim(0, 1.05)
ax.set_ylim(0, 0.95)
ax.axis('off')
ax.set_title('编译器各阶段 (Compiler Stages)', fontsize=14, fontweight='bold', pad=10)
plt.tight_layout()


---
## 3. 语法图 (Syntax Diagrams)

### 什么是语法图？

语法图是一种**图形化**的方式来表示编程语言的语法规则。也叫**铁路图(Railroad Diagram)**。

**为什么用语法图？**
- 比文字描述更直观
- 可以清楚地看到哪些路径是合法的
- 循环和可选部分一目了然

**规则：**
- **圆角矩形/圆形** = 终结符(Terminal) -- 实际出现在代码中的字符/记号
- **方角矩形** = 非终结符(Non-terminal) -- 需要进一步展开的语法规则
- 沿箭头方向走，经过的终结符组成合法的代码

In [ ]:
# 可视化：语法图 - 整数定义
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# === 语法图1: 整数 (Integer) ===
ax = axes[0]
ax.set_title('语法图: 整数 (Integer)', fontsize=12, fontweight='bold')

# 起始箭头
ax.annotate('', xy=(0.1, 0.5), xytext=(0.02, 0.5),
            arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# 可选的符号 + 或 -
# 上方路径（无符号）
ax.annotate('', xy=(0.28, 0.7), xytext=(0.1, 0.7),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))
ax.plot([0.1, 0.1], [0.5, 0.7], 'k-', lw=1.5)
ax.plot([0.28, 0.28], [0.5, 0.7], 'k-', lw=1.5)

# 下方路径（有符号）
# + 号
circle_plus = plt.Circle((0.15, 0.5), 0.04, color='#3498db', zorder=5)
ax.add_patch(circle_plus)
ax.text(0.15, 0.5, '+', ha='center', va='center', fontsize=12, fontweight='bold', color='white', zorder=6)

# - 号
circle_minus = plt.Circle((0.22, 0.5), 0.04, color='#3498db', zorder=5)
ax.add_patch(circle_minus)
ax.text(0.22, 0.5, '-', ha='center', va='center', fontsize=12, fontweight='bold', color='white', zorder=6)

ax.annotate('', xy=(0.18, 0.5), xytext=(0.15 + 0.04, 0.5),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

# 路径选择
ax.plot([0.1, 0.15 - 0.04], [0.5, 0.5], 'k-', lw=1.5)
ax.plot([0.10, 0.10], [0.3, 0.5], 'k-', lw=1.5)
ax.plot([0.10, 0.22 - 0.04], [0.3, 0.3], 'k-', lw=1.5)
ax.plot([0.22 - 0.04, 0.22 - 0.04], [0.3, 0.5 - 0.04], 'k-', lw=1.5)

ax.annotate('', xy=(0.28, 0.5), xytext=(0.26, 0.5),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

# digit 方框（非终结符）- 循环
rect = patches.FancyBboxPatch((0.32, 0.4), 0.14, 0.2,
                               boxstyle='round,pad=0.03',
                               facecolor='#f39c12', alpha=0.85, edgecolor='white', lw=2)
ax.add_patch(rect)
ax.text(0.39, 0.5, 'digit', ha='center', va='center',
        fontsize=11, fontweight='bold', color='white')

# 循环箭头（可以有多个digit）
ax.annotate('', xy=(0.32, 0.5), xytext=(0.28, 0.5),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))
ax.annotate('', xy=(0.55, 0.5), xytext=(0.46, 0.5),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

# 循环回路
ax.plot([0.46, 0.50], [0.5, 0.5], 'k-', lw=1.5)
ax.plot([0.50, 0.50], [0.5, 0.25], 'k-', lw=1.5)
ax.plot([0.50, 0.30], [0.25, 0.25], 'k-', lw=1.5)
ax.plot([0.30, 0.30], [0.25, 0.5], 'k-', lw=1.5)
ax.annotate('', xy=(0.32, 0.35), xytext=(0.44, 0.25),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.2))

# 结束
ax.annotate('', xy=(0.62, 0.5), xytext=(0.55, 0.5),
            arrowprops=dict(arrowstyle='->', color='#333', lw=2))

ax.text(0.7, 0.5, '合法示例:\n+42, -7, 100', fontsize=10, va='center',
        bbox=dict(boxstyle='round', facecolor='#ecf0f1'))

ax.set_xlim(0, 1)
ax.set_ylim(0.1, 0.85)
ax.axis('off')

# === 语法图2: if语句 ===
ax = axes[1]
ax.set_title('语法图: IF语句 (IF Statement)', fontsize=12, fontweight='bold')

elements = [
    (0.05, 'if', '#3498db', 'terminal'),
    (0.17, 'condition', '#f39c12', 'nonterminal'),
    (0.32, 'then', '#3498db', 'terminal'),
    (0.44, 'statement', '#f39c12', 'nonterminal'),
]

for x, label, color, kind in elements:
    w = 0.10 if kind == 'terminal' else 0.12
    if kind == 'terminal':
        circle = patches.FancyBboxPatch((x, 0.35), w, 0.3,
                                         boxstyle='round,pad=0.05',
                                         facecolor=color, alpha=0.85, edgecolor='white', lw=2)
    else:
        circle = patches.Rectangle((x, 0.35), w, 0.3,
                                    facecolor=color, alpha=0.85, edgecolor='white', lw=2)
    ax.add_patch(circle)
    ax.text(x + w/2, 0.5, label, ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')

# else分支（可选路径）
ax.plot([0.56, 0.56], [0.5, 0.15], 'k-', lw=1.5)
ax.plot([0.56, 0.62], [0.15, 0.15], 'k-', lw=1.5)
else_box = patches.FancyBboxPatch((0.62, 0.05), 0.08, 0.2,
                                   boxstyle='round,pad=0.03',
                                   facecolor='#3498db', alpha=0.85, edgecolor='white', lw=2)
ax.add_patch(else_box)
ax.text(0.66, 0.15, 'else', ha='center', va='center',
        fontsize=10, fontweight='bold', color='white')

stmt_box = patches.Rectangle((0.72, 0.05), 0.12, 0.2,
                              facecolor='#f39c12', alpha=0.85, edgecolor='white', lw=2)
ax.add_patch(stmt_box)
ax.text(0.78, 0.15, 'statement', ha='center', va='center',
        fontsize=10, fontweight='bold', color='white')

ax.plot([0.84, 0.88], [0.15, 0.15], 'k-', lw=1.5)
ax.plot([0.88, 0.88], [0.15, 0.5], 'k-', lw=1.5)

# 直通路径
ax.plot([0.56, 0.88], [0.5, 0.5], 'k-', lw=1.5)

# 箭头连接
for i in range(len(elements) - 1):
    x1 = elements[i][0] + (0.10 if elements[i][3] == 'terminal' else 0.12)
    x2 = elements[i+1][0]
    ax.annotate('', xy=(x2, 0.5), xytext=(x1, 0.5),
                arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

ax.annotate('', xy=(0.92, 0.5), xytext=(0.88, 0.5),
            arrowprops=dict(arrowstyle='->', color='#333', lw=2))

ax.set_xlim(0, 1)
ax.set_ylim(-0.05, 0.85)
ax.axis('off')

# 图例
ax2 = axes[2]
ax2.set_title('语法图图例 (Syntax Diagram Legend)', fontsize=12, fontweight='bold')

# 终结符
t_box = patches.FancyBboxPatch((0.1, 0.5), 0.15, 0.3,
                                boxstyle='round,pad=0.05',
                                facecolor='#3498db', alpha=0.85, edgecolor='white', lw=2)
ax2.add_patch(t_box)
ax2.text(0.175, 0.65, 'terminal', ha='center', va='center',
         fontsize=10, fontweight='bold', color='white')
ax2.text(0.175, 0.35, '终结符\n(圆角方框)\n直接出现在代码中', ha='center', va='center', fontsize=9)

# 非终结符
nt_box = patches.Rectangle((0.4, 0.5), 0.18, 0.3,
                            facecolor='#f39c12', alpha=0.85, edgecolor='white', lw=2)
ax2.add_patch(nt_box)
ax2.text(0.49, 0.65, 'non-terminal', ha='center', va='center',
         fontsize=10, fontweight='bold', color='white')
ax2.text(0.49, 0.35, '非终结符\n(直角方框)\n需要进一步展开定义', ha='center', va='center', fontsize=9)

# 箭头
ax2.annotate('', xy=(0.85, 0.65), xytext=(0.72, 0.65),
             arrowprops=dict(arrowstyle='->', color='#333', lw=2.5))
ax2.text(0.785, 0.35, '方向\n(沿箭头方向走)', ha='center', va='center', fontsize=9)

ax2.set_xlim(0, 1)
ax2.set_ylim(0.1, 0.95)
ax2.axis('off')

plt.tight_layout()


---
## 4. BNF范式 (Backus-Naur Form)

### 什么是BNF？

BNF是一种**文本化**的方式来定义语法规则。它和语法图表达的信息相同，但用文字而非图形。

**为什么学BNF？**
- 它是描述编程语言语法的**标准方式**
- 编译器的语法分析器通常基于BNF规则构建
- 考试中经常需要根据BNF判断表达式是否合法

### BNF符号

| 符号 | 含义 | 示例 |
|------|------|------|
| `::=` | 定义为 | `<digit> ::= 0|1|2|...|9` |
| `|` | 或（可选其一） | `<sign> ::= +|-` |
| `<...>` | 非终结符 | `<expression>` |
| 无括号 | 终结符 | `if`, `+`, `(` |

### 示例：定义整数

```
<integer> ::= <sign><digit_sequence> | <digit_sequence>
<sign> ::= + | -
<digit_sequence> ::= <digit> | <digit><digit_sequence>
<digit> ::= 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9
```

### 示例：定义简单算术表达式

```
<expression> ::= <term> | <expression> <addop> <term>
<term> ::= <factor> | <term> <mulop> <factor>
<factor> ::= <number> | <variable> | ( <expression> )
<addop> ::= + | -
<mulop> ::= * | /
<number> ::= <digit> | <digit><number>
<digit> ::= 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9
<variable> ::= <letter> | <letter><variable>
<letter> ::= a | b | c | ... | z
```

**注意递归定义：** `<expression>` 的定义中包含了 `<expression>` 自身，这就是**递归(Recursion)**。
这使得 BNF 可以描述无限长的合法表达式。

In [ ]:
# Python实现：基于BNF的简单语法检查器

class BNFValidator:
    """
    验证字符串是否符合以下BNF规则：
    <integer> ::= <sign><digits> | <digits>
    <sign> ::= + | -
    <digits> ::= <digit> | <digit><digits>
    <digit> ::= 0|1|2|3|4|5|6|7|8|9
    """

    def __init__(self, text):
        self.text = text
        self.pos = 0

    def validate_integer(self):
        """验证<integer>规则"""
        if self.pos >= len(self.text):
            return False

        # 尝试匹配可选的符号
        if self.text[self.pos] in '+-':
            self.pos += 1

        # 必须有至少一个数字
        if not self.validate_digits():
            return False

        # 必须消耗完所有字符
        return self.pos == len(self.text)

    def validate_digits(self):
        """验证<digits>规则：一个或多个digit"""
        if self.pos >= len(self.text) or not self.text[self.pos].isdigit():
            return False
        while self.pos < len(self.text) and self.text[self.pos].isdigit():
            self.pos += 1
        return True

# 测试
test_strings = ['42', '+100', '-7', '3.14', 'abc', '+', '007', '-0', '']

print('BNF整数验证结果:')
print(f'{"输入":<10} {"合法?":<8} {"说明"}')
print('-' * 40)
for s in test_strings:
    v = BNFValidator(s)
    result = v.validate_integer()
    explanation = {
        '42': '无符号正整数',
        '+100': '有正号的整数',
        '-7': '负整数',
        '3.14': '含小数点，不符合integer规则',
        'abc': '字母不是digit',
        '+': '符号后没有数字',
        '007': '合法(BNF允许前导零)',
        '-0': '合法(BNF没有禁止-0)',
        '': '空字符串',
    }


---
## 5. 逆波兰表达式 (Reverse Polish Notation, RPN)

### 5.1 什么是RPN？

我们通常写的算术表达式是**中缀表达式(Infix Notation)**：运算符在操作数之间。

```
中缀: 3 + 4
```

**逆波兰表达式(RPN/后缀表达式, Postfix Notation)**：运算符在操作数之后。

```
后缀: 3 4 +
```

### 5.2 为什么RPN重要？

1. **不需要括号！** -- 运算顺序由表达式结构决定
   - 中缀: `(3 + 4) * 2` 需要括号
   - RPN: `3 4 + 2 *` 不需要括号

2. **计算机处理更简单** -- 用栈(Stack)就能求值，不需要考虑优先级

3. **编译器中间表示** -- 很多编译器将表达式转为RPN作为中间步骤

### 5.3 中缀转RPN的规则 (Shunting-Yard Algorithm)

使用Dijkstra的调度场算法(Shunting-Yard Algorithm)：

1. 从左到右扫描中缀表达式
2. 如果是**数字** -> 直接输出
3. 如果是**运算符**:
   - 将栈中优先级 >= 当前运算符的运算符弹出并输出
   - 将当前运算符压入栈
4. 如果是**左括号** `(` -> 压入栈
5. 如果是**右括号** `)` -> 弹出栈中运算符直到遇到左括号
6. 扫描完成后，弹出栈中所有运算符

### 更多转换示例

| 中缀表达式 | RPN表达式 |
|------------|----------|
| `3 + 4` | `3 4 +` |
| `3 + 4 * 2` | `3 4 2 * +` |
| `(3 + 4) * 2` | `3 4 + 2 *` |
| `a + b * c - d` | `a b c * + d -` |
| `(a + b) * (c - d)` | `a b + c d - *` |

In [ ]:
# Python实现：中缀表达式转RPN (Shunting-Yard Algorithm)

def infix_to_rpn(expression):
    """将中缀表达式转换为逆波兰表达式"""
    precedence = {'+': 1, '-': 1, '*': 2, '/': 2, '^': 3}
    right_assoc = {'^'}  # 右结合运算符
    output = []
    op_stack = []
    steps = []  # 记录每一步

    # 分割表达式为记号
    tokens = re.findall(r'\d+\.?\d*|[a-zA-Z]+|[+\-*/^()]', expression)

    for token in tokens:
        if re.match(r'\d+\.?\d*|[a-zA-Z]+', token):
            # 数字或变量 -> 直接输出
            output.append(token)
            steps.append((token, '数字/变量->输出', list(op_stack), list(output)))

        elif token in precedence:
            # 运算符
            while (op_stack and
                   op_stack[-1] != '(' and
                   op_stack[-1] in precedence and
                   (precedence[op_stack[-1]] > precedence[token] or
                    (precedence[op_stack[-1]] == precedence[token] and token not in right_assoc))):
                output.append(op_stack.pop())
            op_stack.append(token)
            steps.append((token, '运算符->栈', list(op_stack), list(output)))

        elif token == '(':
            op_stack.append(token)
            steps.append((token, '左括号->栈', list(op_stack), list(output)))

        elif token == ')':
            while op_stack and op_stack[-1] != '(':
                output.append(op_stack.pop())
            if op_stack:
                op_stack.pop()  # 弹出左括号
            steps.append((token, '右括号:弹出到(', list(op_stack), list(output)))

    while op_stack:
        output.append(op_stack.pop())

    steps.append(('END', '清空栈', list(op_stack), list(output)))
    return ' '.join(output), steps

# 测试多个表达式
expressions = [
    '3 + 4',
    '3 + 4 * 2',
    '( 3 + 4 ) * 2',
    'a + b * c - d',
    '( a + b ) * ( c - d )',
]

print('中缀表达式 -> RPN转换结果:')
print(f'{"中缀 (Infix)":<25} {"后缀 (RPN/Postfix)"}')
print('-' * 50)
for expr in expressions:
    rpn, _ = infix_to_rpn(expr)


In [ ]:
# 可视化：中缀转RPN的详细过程
expr = '( 3 + 4 ) * 2'
rpn, steps = infix_to_rpn(expr)

print(f'表达式: {expr}')
print(f'RPN结果: {rpn}')
print()
print(f'{"步骤":<5} {"读取":<8} {"动作":<18} {"运算符栈":<15} {"输出"}')
print('-' * 65)
for i, (token, action, stack, output) in enumerate(steps, 1):
    stack_str = str(stack) if stack else '[]'
    output_str = ' '.join(output) if output else '(empty)'


In [ ]:
# 可视化：中缀转RPN的过程动画（静态版）
expr = '( 3 + 4 ) * 2'
_, steps = infix_to_rpn(expr)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

# 只显示关键步骤
key_steps = steps[:min(8, len(steps))]

for idx, (token, action, stack, output) in enumerate(key_steps):
    ax = axes[idx]

    # 标题
    ax.set_title(f'步骤{idx+1}: 读取 "{token}"', fontsize=10, fontweight='bold')

    # 运算符栈
    ax.text(0.25, 0.95, '运算符栈', ha='center', va='center', fontsize=9, fontweight='bold', color='#e74c3c')
    for i, item in enumerate(stack):
        y = 0.15 + i * 0.15
        rect = patches.FancyBboxPatch((0.05, y), 0.4, 0.12,
                                       boxstyle='round,pad=0.02',
                                       facecolor='#e74c3c', alpha=0.7, edgecolor='white')
        ax.add_patch(rect)
        ax.text(0.25, y + 0.06, item, ha='center', va='center',
                fontsize=12, fontweight='bold', color='white')

    # 输出队列
    ax.text(0.75, 0.95, '输出', ha='center', va='center', fontsize=9, fontweight='bold', color='#2ecc71')
    output_text = ' '.join(output) if output else '(empty)'
    ax.text(0.75, 0.5, output_text, ha='center', va='center',
            fontsize=11, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#2ecc71', alpha=0.2))

    # 动作说明
    ax.text(0.5, 0.02, action, ha='center', va='center', fontsize=8, color='#666', style='italic')

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

plt.suptitle(f'中缀转RPN过程: {expr} -> {" ".join(steps[-1][3])}',
             fontsize=13, fontweight='bold')
plt.tight_layout()


### 5.4 RPN求值

RPN表达式的求值非常简单，只需要一个栈：

1. 从左到右扫描RPN表达式
2. 如果是**数字** -> 压入栈
3. 如果是**运算符** -> 从栈中弹出两个操作数，计算结果，将结果压回栈
4. 最终栈中的唯一元素就是结果

In [ ]:
# Python实现：RPN计算器

def evaluate_rpn(expression):
    """求值逆波兰表达式"""
    stack = []
    steps = []
    tokens = expression.split()

    for token in tokens:
        if token in '+-*/':
            b = stack.pop()  # 注意顺序：先弹出的是第二个操作数
            a = stack.pop()
            if token == '+':
                result = a + b
            elif token == '-':
                result = a - b
            elif token == '*':
                result = a * b
            elif token == '/':
                result = a / b
            stack.append(result)
            steps.append((token, f'{a} {token} {b} = {result}', list(stack)))
        else:
            stack.append(float(token))
            steps.append((token, f'压入 {token}', list(stack)))

    return stack[0], steps

# 测试RPN求值
rpn_expressions = [
    ('3 4 +', '3 + 4'),
    ('3 4 2 * +', '3 + 4 * 2'),
    ('3 4 + 2 *', '(3 + 4) * 2'),
    ('5 1 2 + 4 * + 3 -', '5 + (1 + 2) * 4 - 3'),
]

for rpn, infix in rpn_expressions:
    result, steps = evaluate_rpn(rpn)
    print(f'中缀: {infix}')
    print(f'RPN:  {rpn}')
    print(f'步骤:')
    for token, desc, stack in steps:
        print(f'  读取 {token:>3} | {desc:<20} | 栈: {stack}')
    print(f'结果: {result}')


In [ ]:
# 可视化：RPN求值过程
rpn = '3 4 + 2 *'
result, steps = evaluate_rpn(rpn)

fig, axes = plt.subplots(1, len(steps), figsize=(3 * len(steps), 5))

for idx, (token, desc, stack) in enumerate(steps):
    ax = axes[idx]
    ax.set_title(f'步骤{idx+1}\n读取: {token}', fontsize=10, fontweight='bold')

    # 画栈
    max_height = 4
    # 栈底线
    ax.plot([0.2, 0.8], [0.1, 0.1], 'k-', lw=3)
    ax.plot([0.2, 0.2], [0.1, 0.9], 'k-', lw=2)
    ax.plot([0.8, 0.8], [0.1, 0.9], 'k-', lw=2)

    for i, val in enumerate(stack):
        y = 0.12 + i * 0.18
        color = '#3498db' if i < len(stack) - 1 else '#e74c3c'
        rect = patches.FancyBboxPatch((0.22, y), 0.56, 0.15,
                                       boxstyle='round,pad=0.02',
                                       facecolor=color, alpha=0.8, edgecolor='white', lw=2)
        ax.add_patch(rect)
        display = int(val) if val == int(val) else val
        ax.text(0.5, y + 0.075, str(display), ha='center', va='center',
                fontsize=14, fontweight='bold', color='white')

    # 操作描述
    is_op = token in '+-*/'
    if is_op:
        ax.text(0.5, 0.92, desc, ha='center', va='center', fontsize=8, color='#e74c3c',
                fontweight='bold')
    else:
        ax.text(0.5, 0.92, desc, ha='center', va='center', fontsize=8, color='#3498db')

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

plt.suptitle(f'RPN求值过程: {rpn} = {int(result)}',
             fontsize=13, fontweight='bold')
plt.tight_layout()


---
## 6. 综合练习 (Exercises)

### 练习1：编译阶段

1. 列出编译器的四个主要阶段，并简述每个阶段的作用。
2. 解释为什么编译器需要多次遍历(multiple passes)源代码。
3. 编译器和解释器各适合什么场景？举例说明。

### 练习2：词法分析

对以下代码进行词法分析，列出所有记号及其类型：

```python
if count >= 10:
    result = count * 2 + 1
```

### 练习3：BNF

给定以下BNF规则：
```
<assignment> ::= <variable> = <expression>
<expression> ::= <term> | <expression> + <term> | <expression> - <term>
<term> ::= <factor> | <term> * <factor> | <term> / <factor>
<factor> ::= <number> | <variable> | ( <expression> )
<number> ::= <digit> | <digit><number>
<digit> ::= 0|1|2|3|4|5|6|7|8|9
<variable> ::= a|b|c|d|x|y|z
```

判断以下表达式是否合法：
1. `x = 5 + 3`
2. `y = (a + b) * c`
3. `z = a + * b`
4. `x = ((2 + 3))`

### 练习4：RPN转换与求值

In [ ]:
# 练习2：对代码进行词法分析
code = 'if count >= 10:\n    result = count * 2 + 1'
print('源代码:')
print(code)
print()

# 用我们的词法分析器验证
lexer = SimpleLexer('if count >= 10')
tokens = lexer.tokenize()
print('第1行记号:')
for t in tokens:
    print(f'  {t}')

print()
lexer = SimpleLexer('result = count * 2 + 1')
tokens = lexer.tokenize()
print('第2行记号:')
for t in tokens:


In [ ]:
# 练习4：RPN转换与求值
print('将以下中缀表达式转换为RPN，然后求值：')
print()

exercises = [
    '5 + 3 * 2',
    '( 5 + 3 ) * 2',
    '8 / 4 + 6 * 2',
    '( 8 + 2 ) * ( 5 - 3 )',
]

for expr in exercises:
    rpn, _ = infix_to_rpn(expr)
    result, _ = evaluate_rpn(rpn)
    display_result = int(result) if result == int(result) else result
    print(f'中缀: {expr}')
    print(f'RPN:  {rpn}')
    print(f'结果: {display_result}')


### 练习5：手动转换

不使用代码，手动将以下中缀表达式转换为RPN：

1. `a * b + c`
2. `a * ( b + c )`
3. `a + b * c - d / e`
4. `( a + b ) * ( c + d ) - e`

然后用下面的代码验证你的答案：

In [ ]:
# 验证练习5的答案
ex5 = [
    'a * b + c',
    'a * ( b + c )',
    'a + b * c - d / e',
    '( a + b ) * ( c + d ) - e',
]

print('练习5参考答案:')
for expr in ex5:
    rpn, _ = infix_to_rpn(expr)


### 练习6：编程挑战

1. 扩展词法分析器，使其能处理注释（`#` 后面的内容是注释）
2. 实现一个完整的计算器：输入中缀表达式字符串，输出计算结果
   - 支持 +, -, *, / 四则运算
   - 支持括号
   - 支持小数

In [ ]:
# 练习6 - 完整计算器实现提示
def calculator(expression):
    """
    完整计算器：中缀表达式 -> RPN -> 求值
    """
    # 步骤1: 中缀转RPN
    rpn, _ = infix_to_rpn(expression)
    # 步骤2: RPN求值
    result, _ = evaluate_rpn(rpn)
    return result

# 测试计算器
test_expressions = [
    '( 3 + 4 ) * ( 5 - 2 )',
    '10 / 2 + 8 * 3',
    '( 1 + 2 ) * ( 3 + 4 ) * ( 5 + 6 )',
]

for expr in test_expressions:
    result = calculator(expr)
    display = int(result) if result == int(result) else result


---
## 本节总结 (Summary)

1. **翻译软件**分为汇编器、编译器和解释器，各有适用场景
2. **编译过程**分为多个阶段：词法分析 -> 语法分析 -> 语义分析 -> 代码优化 -> 代码生成
3. **词法分析**将源代码分解为记号(Tokens)
4. **语法分析**检查记号序列是否符合语法规则，构建语法树(AST)
5. **语法图**和**BNF**是描述语言语法的两种方式（图形 vs 文本）
6. **逆波兰表达式(RPN)**：
   - 不需要括号即可表达运算优先级
   - 中缀->RPN使用调度场算法(Shunting-Yard)
   - RPN求值使用栈
   - 编译器常用RPN作为中间表示

---
*Cambridge A-Level Computer Science (9618) - Chapter 16*